# 재료정보학 실습

**Materials Informatics · MI · 소재정보학**

소재 데이터와 통계·계산·머신러닝을 연결해 소재를 이해하고 설계하는 분야.

소재 분야에서 이해하기: 실험과 계산 데이터를 모아 조성–물성 관계를 분석한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [소재 발견용 그래프 신경망 연구](https://www.nature.com/articles/s41586-023-06735-9)

## 1. 데이터에서 조성–물성 관계 찾기

실험과 계산 데이터를 한 표로 모아 관계를 분석하는 전형적인 흐름을 따라갑니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

import pandas as pd

properties = {'Li': (0.98, 1.52), 'Mg': (1.31, 1.60), 'Al': (1.61, 1.43),
              'Ti': (1.54, 1.47), 'Fe': (1.83, 1.26), 'Ni': (1.91, 1.24)}   # (전기음성도, 원자반지름 A)
elements = list(properties)

rows = []
for index in range(260):
    picked = rng.choice(elements, 3, replace=False)
    weights = rng.dirichlet([2, 2, 2])
    electro = sum(w * properties[e][0] for w, e in zip(weights, picked))
    radius = sum(w * properties[e][1] for w, e in zip(weights, picked))
    hardness = 90 + 130 * electro - 60 * radius ** 2 + rng.normal(0, 6)
    rows.append({'composition': '-'.join('%s%.2f' % (e, w) for e, w in zip(picked, weights)),
                 'mean_electronegativity': electro, 'mean_radius': radius,
                 'source': 'experiment' if index % 3 == 0 else 'calculation',
                 'hardness_HV': hardness})
table = pd.DataFrame(rows)
print(table.head())
print('\n출처별 건수:'); print(table.source.value_counts())

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

features = table[['mean_electronegativity', 'mean_radius']].to_numpy()
target = table.hardness_HV.to_numpy()
score = cross_val_score(RandomForestRegressor(n_estimators=300, random_state=0),
                        features, target, cv=5, scoring='r2').mean()
print('교차검증 R2 %.3f' % score)

scatter = plt.scatter(table.mean_electronegativity, table.mean_radius, c=target, cmap='viridis', s=18)
plt.colorbar(scatter, label='hardness (HV)')
plt.xlabel('mean electronegativity'); plt.ylabel('mean atomic radius (A)'); plt.show()

## 2. 출처가 섞인 데이터의 함정

In [ ]:
table.loc[table.source == 'calculation', 'hardness_HV'] += 25       # 계산이 계통적으로 높게 나온 상황
biased = table.hardness_HV.to_numpy()
print('출처를 무시한 R2 %.3f'
      % cross_val_score(RandomForestRegressor(n_estimators=300, random_state=0), features, biased, cv=5, scoring='r2').mean())
with_source = np.column_stack([features, (table.source == 'calculation').to_numpy(float)])
print('출처를 변수로 넣은 R2 %.3f'
      % cross_val_score(RandomForestRegressor(n_estimators=300, random_state=0), with_source, biased, cv=5, scoring='r2').mean())
print('\n출처별 계통 차이를 모델에 알려주지 않으면 그 차이가 잡음으로 들어갑니다.')
print('데이터를 모으는 일이 재료정보학의 실제 작업 대부분을 차지합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#materials-informatics)을 여세요.